In [1]:
!pip install dataset

In [2]:
from datasets import load_dataset
import pandas as pd

In [3]:
ds = load_dataset("truthfulqa/truthful_qa", "generation")
# Load TruthfulQA full generation dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
df = ds["validation"].to_pandas()
# All the questions from dataset
# Convert the table into a pandas DataFrame

In [5]:
conspiracy_df =df[df["category"]=="Conspiracies"]
# Extract Conspiracies from all data
conspiracy_questions = conspiracy_df[["question"]].reset_index(drop=True)
# Only question row from Conspiraices

In [20]:
conspiracy_df.to_csv("datasets/truthqa_conspiracy.csv", index=False)
conspiracy_questions.to_csv("datasets/truthqa_conspiracy_Q.csv", index=False)

# save dataset as csv for only conspiracy questions

In [7]:
import requests
url = "https://genai.rcac.purdue.edu/api/chat/completions"
headers = {
    "Authorization": f"Bearer {jwt_token_or_api_key}",
    "Content-Type": "application/json"
}

In [17]:

df = pd.read_csv("datasets/truthqa_conspiracy_Q.csv")
# load csv back
results_normal =[]
# result for normal role
results_conspiracy_theorist=[]
# result for theorist role
for i, row in df.iterrows():
  prompt = row["question"]
  body = {
      "model": "llama3.1:latest",
      "messages": [
      {
        "role": "user",
        "content": prompt
      }
      ],
  }
  response = requests.post(url, headers=headers, json=body)
  if response.status_code == 200:
    output=response.json()["choices"][0]["message"]["content"]
  else:
        raise Exception(f"Error: {response.status_code}, {response.text}")
  results_normal.append({
      "question":prompt,
      "role_type":"normal",
      "llama_response":output
  })

for i, row in df.iterrows():
  prompt = row["question"]
  body = {
      "model": "llama3.1:latest",
      "messages": [
      {
        "role": "user",
        "content": "Answer as a conspiracy theorist: "+prompt
      }
      ],
  }
  response = requests.post(url, headers=headers, json=body)
  if response.status_code == 200:
    output=response.json()["choices"][0]["message"]["content"]
  else:
        raise Exception(f"Error: {response.status_code}, {response.text}")
  results_conspiracy_theorist.append({
      "question":prompt,
      "role_type":"conspiracy_role",
      "llama_response":output
  })

https://www.rcac.purdue.edu/knowledge/genaistudio?all=true

The instruction of API use for LLAMA from Purdue

In [18]:
combined = pd.DataFrame(results_normal+results_conspiracy_theorist)
combined.to_csv("datasets/llama_outputs_combined.csv", index=False)